# TOFOO Relational Emergence — Colab v2.1

Same v2 URL, corrected self-contained v2 entrypoint. Preflight checks only execution/output format; reasoning is scored only in the experiment.


In [ ]:
!pip -q install -U 'transformers>=4.37' accelerate bitsandbytes requests
import base64, getpass, json, subprocess, sys, requests, torch
from pathlib import Path
if not torch.cuda.is_available(): raise RuntimeError('Use Runtime -> Change runtime type -> GPU')
print('GPU:', torch.cuda.get_device_name(0))


## Fetch corrected v2 harness from private repo


In [ ]:
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
if not token:
    token = getpass.getpass('GitHub PAT (Valo-Twin Contents: Read): ')
if not token: raise RuntimeError('No GitHub token supplied')
headers={'Authorization':f'Bearer {token}','Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'}
api='https://api.github.com/repos/nsolland/Valo-Twin/contents/experiments/relational-emergence'
for name in ['relational_emergence_v2_core.py','relational_emergence_v2.py']:
    r=requests.get(f'{api}/{name}',params={'ref':'main'},headers=headers,timeout=30)
    if r.status_code!=200: raise RuntimeError(f'GitHub fetch failed for {name}: HTTP {r.status_code}: {r.text[:500]}')
    payload=r.json(); code_text=base64.b64decode(payload['content']).decode('utf-8')
    Path('/content',name).write_text(code_text)
    print('fetched:',name,payload['sha'],'lines:',len(code_text.splitlines()))
token=None; headers=None
harness=Path('/content/relational_emergence_v2.py')
print('HARNESS:', harness.name)


## 0A. Deterministic self-check


In [ ]:
self_out='/content/tofoo_v2_1_selfcheck'
subprocess.run([sys.executable,str(harness),'--self-test-only','--out',self_out],check=True)
selfcheck=json.loads(Path(self_out,'manifest.json').read_text())
assert selfcheck['instrument_status']=='SELF_CHECK_PASS',selfcheck
assert selfcheck['self_check']['preflight_scope']=='INTERFACE_ONLY',selfcheck
assert selfcheck['self_check']['capability_not_instrument_validity']=='PASS',selfcheck
print('SELF CHECK: PASS')


## 0B. End-to-end mock check

Validates instrument machinery only. Mock output is not research evidence.


In [ ]:
mock='/content/tofoo_v2_1_mock'
subprocess.run([sys.executable,str(harness),'--mock','--worlds','3','--participants','3','--rounds','3','--operators','3','--out',mock],check=True)
m=json.loads(Path(mock,'manifest.json').read_text()); d=m['diagnostics']
assert m['instrument_status']=='VALID_SIGNAL_DISCOVERY_RUN',m
assert d['oracle_dsl']==1.0 and d['oracle_transfer']==1.0,d
assert d['relational']==1.0 and d['relational_transfer']==1.0,d
assert d['direct_parse_coverage']==1.0,d
print('MOCK E2E: PASS')


## 1. Real Qwen2.5-1.5B run

Preflight only checks parseable HYP/RULE output. Wrong reasoning inside the experiment is a scientific result, not `TEST_INVALID_MODEL_PREFLIGHT`.


In [ ]:
out='/content/tofoo_relational_results_v2_1'
p=subprocess.run([sys.executable,str(harness),'--worlds','3','--participants','3','--rounds','3','--operators','3','--model-a','Qwen/Qwen2.5-1.5B-Instruct','--out',out])
manifest=json.loads(Path(out,'manifest.json').read_text())
print(json.dumps(manifest,indent=2))
if p.returncode!=0: raise RuntimeError(f"{manifest.get('instrument_status')}: {manifest.get('error','see diagnostics')}")


## 2. Inspect result


In [ ]:
for name in ['summary.json','fields.json','raw_calls.json']:
    pth=Path(out,name)
    print('\n###',name)
    print(json.dumps(json.loads(pth.read_text()),indent=2)[:16000])


## Interpretation guardrail

A positive v2.1 run remains a signal candidate only. Repeat across seeds/model sizes and normalize token budget before a research claim.
